# batchnorm-running-stats — faded example 2: Manually apply eval-mode BN normalization (complete the standardize)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `batchnorm-running-stats`. Running the beacon reports progress on the `CNN: BatchNorm running stats` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: BatchNorm running stats` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`batchnorm-running-stats`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "batchnorm-running-stats"
DD_SUBTOPIC = "CNN: BatchNorm running stats"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Eval-mode BatchNorm applies `y = weight * (x - running_mean) / sqrt(running_var + eps) + bias` per channel, using the frozen buffers. The channel-wise tensors must be reshaped to `(1, C, 1, 1)` so they broadcast over the batch and spatial dimensions of a `(B, C, H, W)` input.

## Faded exercise 2

Implement `eval_bn_manual(x, running_mean, running_var, weight, bias, eps)` that reproduces eval-mode `BatchNorm2d` by hand. The reshaping to `(1, C, 1, 1)` and the final affine `weight`/`bias` application are given. You must complete the standardization step: subtract the running mean and divide by the standard deviation derived from the running variance plus eps.

**Fill in:** the standardized tensor (x minus running mean, divided by sqrt of running var plus eps), before the affine weight/bias is applied

In [ ]:
def eval_bn_manual(x, running_mean, running_var, weight, bias, eps=1e-5):
    mean = rearrange(running_mean, 'c -> 1 c 1 1')
    var = rearrange(running_var, 'c -> 1 c 1 1')
    w = rearrange(weight, 'c -> 1 c 1 1')
    b = rearrange(bias, 'c -> 1 c 1 1')
    standardized = (x - mean) / t.sqrt(var + eps)
    return w * standardized + b

t.manual_seed(0)
x = t.randn(2, 3, 4, 4)
rm = t.tensor([0.0, 1.0, -1.0])
rv = t.tensor([1.0, 2.0, 0.5])
wt = t.tensor([1.0, 2.0, 0.5])
bs = t.tensor([0.0, -1.0, 1.0])
y = eval_bn_manual(x, rm, rv, wt, bs)
print("output shape:", tuple(y.shape))
print("y[0,1,0,0]:", round(y[0, 1, 0, 0].item(), 4))


def _test():
    t.manual_seed(0)
    x = t.randn(2, 3, 4, 4)
    rm = t.tensor([0.0, 1.0, -1.0])
    rv = t.tensor([1.0, 2.0, 0.5])
    wt = t.tensor([1.0, 2.0, 0.5])
    bs = t.tensor([0.0, -1.0, 1.0])
    y = eval_bn_manual(x, rm, rv, wt, bs)
    bn = t.nn.BatchNorm2d(3)
    with t.no_grad():
        bn.running_mean.copy_(rm)
        bn.running_var.copy_(rv)
        bn.weight.copy_(wt)
        bn.bias.copy_(bs)
    bn.eval()
    ref = bn(x)
    assert y.shape == ref.shape, "shape mismatch"
    assert t.allclose(y, ref, atol=1e-5), "eval-mode normalization mismatch vs nn.BatchNorm2d"


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def eval_bn_manual(x, running_mean, running_var, weight, bias, eps=1e-5):
    mean = rearrange(running_mean, 'c -> 1 c 1 1')
    var = rearrange(running_var, 'c -> 1 c 1 1')
    w = rearrange(weight, 'c -> 1 c 1 1')
    b = rearrange(bias, 'c -> 1 c 1 1')
    standardized = (x - mean) / t.sqrt(var + eps)
    return w * standardized + b

t.manual_seed(0)
x = t.randn(2, 3, 4, 4)
rm = t.tensor([0.0, 1.0, -1.0])
rv = t.tensor([1.0, 2.0, 0.5])
wt = t.tensor([1.0, 2.0, 0.5])
bs = t.tensor([0.0, -1.0, 1.0])
y = eval_bn_manual(x, rm, rv, wt, bs)
print("output shape:", tuple(y.shape))
print("y[0,1,0,0]:", round(y[0, 1, 0, 0].item(), 4))
```
</details>